# Повторение: временная и пространственная сложность

**Временная сложность** описывает, как растёт число операций при увеличении входных данных. **Пространственная сложность** — как растёт потребность в дополнительной памяти.

Обозначим размер входа через `n`. В O-оценке отбрасываем постоянные множители и медленнее растущие слагаемые: `3n + 5` → O(n), `n² + n` → O(n²). Здесь выбираем наиболее точную оценку роста.

Считаем операции с отдельным числом или короткой строкой и память для них постоянными. Время оцениваем в худшем случае по входным данным; для множеств и словарей отдельно используем среднюю стоимость операций.


## 1. Основные оценки

| Оценка | Пример | Если увеличить `n` вдвое |
|---|---|---|
| O(1) | Индекс списка, `len(items)` | Объём работы не меняется |
| O(log n) | Деление `n` пополам в цикле | Примерно один дополнительный шаг |
| O(n) | Один проход по списку | Примерно вдвое больше работы |
| O(n log n) | Сортировка сравнением в худшем случае | Чуть больше чем вдвое |
| O(n²) | Перебор всех пар элементов | Примерно вчетверо больше работы |

Таблица показывает характер роста, а не точное число операций или секунд.


## 2. Как оценивать код

**Последовательные действия складываем:** два прохода по `n` элементам дают O(n) + O(n) = O(n).

**Для вложенных действий считаем повторы:** если внешний цикл выполняется `n` раз, а внутренний каждый раз `m` раз, получаем O(n · m). Вложенность сама по себе ещё не означает O(n²): внутренний цикл может делать постоянное число шагов.

Учитывай работу внутри встроенных функций:

```python
for x in items:          # n повторений
    count = items.count(x)  # до n проверок каждый раз
# Всего O(n²) времени, хотя явно написан только один цикл.
```

Ранний `return` может ускорить отдельный запуск. Но для худшего случая нужно рассмотреть вход, на котором приходится пройти все данные.


## 3. Стоимость операций Python

| Операция | Время |
|---|---|
| `items[i]`, `len(items)` для списка | O(1) |
| `items.append(x)` | O(1) амортизированно* |
| `x in items`, `items.count(x)`, `sum(items)` для списка | O(n) |
| `items.pop(0)`, `items.insert(0, x)` | O(n): элементы сдвигаются |
| `items[a:b]` | O(k), где `k` — длина среза; создаётся новый список |
| `sorted(items)` | O(n log n) в худшем случае; O(n) дополнительной памяти |
| `x in seen`, `seen.add(x)` для множества | O(1) в среднем |
| `counts[x]`, `counts.get(x, 0)`, `counts[x] = value` для словаря | O(1) в среднем |
| `set(items)` | O(n) в среднем; до O(n) дополнительной памяти |

*Амортизированно: суммарно `n` добавлений в конец списка требуют O(n) времени, хотя отдельное добавление может быть дороже из-за расширения списка.

В `set` можно хранить только хешируемые значения; то же ограничение действует для ключей `dict`. Числа и строки подходят, списки — нет. Множество не сохраняет порядок и убирает повторы: нельзя бездумно заменять им список результата.


## 4. Дополнительная память

В этом занятии **не учитываем вход, но учитываем созданный результат**. Считаем максимальный объём одновременно используемой дополнительной памяти, а не сумму всех выделений за время работы.

```python
# O(n) времени, O(n) дополнительной памяти в худшем случае:
positive = [x for x in numbers if x > 0]
total = sum(positive)

# O(n) времени, O(1) дополнительной памяти:
total = 0
for x in numbers:
    if x > 0:
        total += x
```

Оба варианта считают одну сумму. В первом хранится до `n` чисел, во втором — только текущий элемент и сумма.

Цикл из `n²` шагов не обязательно требует O(n²) памяти: если на каждом шаге обновляется один счётчик, дополнительная память остаётся O(1).

**Генератор** выдаёт элементы по одному. Для простого генератора чисел достаточно O(1) памяти на его состояние, но `list(generator)` сохраняет все выданные значения и требует O(n). Генератор нельзя повторно обойти: сумму и количество нужно вычислять в одном проходе.


## 5. Три полезных приёма

**Много проверок наличия:** один раз построй множество до цикла.

```python
banned_set = set(banned)  # O(m) времени и памяти
allowed = []
for name in names:       # n имён
    if name not in banned_set:
        allowed.append(name)
```

Итого O(n + m) времени в среднем вместо O(n · m) при поиске по списку. Память с учётом ответа — O(n + m) в худшем случае. Порядок и повторы разрешённых имён сохраняются.

**Подсчёт частот:** увеличивай счётчик при каждой встрече значения.

```python
counts = {}
for x in numbers:
    counts[x] = counts.get(x, 0) + 1
```

O(n) времени в среднем и O(k) памяти, где `k` — количество разных значений. Если все числа одинаковые, хранится один счётчик; если разные — `n` счётчиков.

**Нужны только итоговые числа:** обновляй сумму, количество или максимум сразу, вместо того чтобы собирать промежуточный список. Часто так можно получить O(1) дополнительной памяти.


## Перед тем как считать решение готовым

1. Что обозначают `n`, `m`, `k` именно в этой задаче?
2. Сколько раз выполняется дорогая операция? Нет ли `in` по списку или `count` внутри цикла?
3. Какие коллекции создаются и сколько элементов они могут одновременно хранить?
4. Сохранились ли нужные порядок, повторы и исходные данные?

`time.perf_counter()` помогает измерить время, `tracemalloc` — пиковую память выделений Python. Замеры зависят от среды и не доказывают O-оценку: сначала объясни рост по коду, затем проверь наблюдения на разных размерах входа.
